In [ ]:
!pip install transformers onnxruntime onnx psutil matplotlib optimum optimum-onnx

# FP32 (baseline)





In [3]:
from transformers import AutoTokenizer, pipeline, DistilBertForSequenceClassification
import torch

def bert_fp32(model_checkpoint, device):

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = DistilBertForSequenceClassification.from_pretrained(
        model_checkpoint)
    model.to(device)

    # Create the pipeline, specifying the device
    return pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

model_checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
text = "Hello, my dog is cute"

# Determine the device based on CUDA availability
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Call the function with the determined device
pred = bert_fp32(model_checkpoint, device)(text)
print(pred)

Using device: cpu


Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9997830986976624}]


# ONNX Runtime CPU - FP32



In [15]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer, pipeline
import torch
import onnxruntime as ort

def bert_onnx_fp32(model_checkpoint, save_directory, providers, device):
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    ort_model = ORTModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        export=True,
        providers=providers)

    ort_model.save_pretrained(save_directory)
    return pipeline("text-classification", model=ort_model, tokenizer=tokenizer, device=device)

model_checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
save_directory = "onnx/distilled-bert"
text = "Hello, my dog is cute"

# Determine the device based on CUDA availability
device = "cuda:0" if torch.cuda.is_available() else "cpu"
inputs = "Hello, my dog is cute"
if "CUDAExecutionProvider" in ort.get_available_providers() and device == "cuda:0":
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
else:
    providers = ["CPUExecutionProvider"]

pred = bert_onnx_fp32(model_checkpoint, save_directory, providers, device)(text)
pred

The model distilbert/distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9997830986976624}]

# ONNX Runtime CPU - Dynamic - INT8


In [13]:
from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification, pipeline
from optimum.onnxruntime.configuration import AutoQuantizationConfig

def bert_quantize(model_checkpoint, save_directory, providers, device):
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    onnx_model = ORTModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        export=True,
        providers = providers)

    # Create quantizer from model
    quantizer = ORTQuantizer.from_pretrained(onnx_model)

    # Define the dynamic quantization strategy by creating the configuration
    dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

    # Quantize the model
    quantizer.quantize(
        save_dir=save_directory,
        quantization_config=dqconfig
        )

    # inference
    model_dyn_u8s8 = ORTModelForSequenceClassification.from_pretrained(
        save_directory,
        file_name="model_quantized.onnx",
        providers = providers)

    return pipeline("text-classification", model=model_dyn_u8s8, tokenizer=tokenizer, device=device)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
inputs = "Hello, my dog is cute"
if "CUDAExecutionProvider" in ort.get_available_providers() and device == "cuda:0":
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
else:
    providers = ["CPUExecutionProvider"]

model_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
int8_save_dir = "onnx/int8-distilled-bert"
pred = bert_quantize(model_checkpoint, int8_save_dir, providers, device)(text)
print(pred)


The model distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.99978107213974}]


## Optimizing with ORTOptimizer

four common optimization levels:
* O1: basic general optimizations.
* O2: basic and extended general optimizations, transformers-specific fusions.
* O3: same as O2 with GELU approximation.
* O4: same as O3 with mixed precision (fp16, GPU-only).

In [ ]:
from transformers import AutoTokenizer
from optimum.onnxruntime import (
    ORTModelForSequenceClassification,
    ORTOptimizer
)
from optimum.onnxruntime.configuration import AutoOptimizationConfig
from optimum.onnxruntime import pipeline

def bert_optimize(mode_dir, save_directory, providers, device):
    # Load the tokenizer and export the model to the ONNX format
    #model_id = "distilbert-base-uncased-finetuned-sst-2-english"

    tokenizer = AutoTokenizer.from_pretrained(mode_dir)
    model = ORTModelForSequenceClassification.from_pretrained(
        mode_dir,
        file_name="model_quantized.onnx",
        providers = providers)

    # Load the optimization configuration detailing the optimization we wish to apply
    optimization_config = AutoOptimizationConfig.O3()
    optimizer = ORTOptimizer.from_pretrained(model)

    optimizer.optimize(save_dir=save_directory, optimization_config=optimization_config)
    optimized_model = ORTModelForSequenceClassification.from_pretrained(
        optimize_save_dir,
        file_name="model_optimized.onnx",
        providers=providers)

    # Create the transformers pipeline
    return pipeline("text-classification", model=optimized_model, tokenizer=tokenizer, device=device)

    # Save and push the model to the hub
    #model.save_pretrained("../onnx/optimizer-distilled-bert/")

mode_dir = "onnx/int8-distilled-bert"
optimize_save_dir = "onnx/optimize-distilled-bert"
text = "I like the new ORT pipeline"
pred = bert_optimize(mode_dir, optimize_save_dir, providers, device)(text)
print(pred)

In [16]:
import time, os
import torch
import pandas as pd
from transformers import AutoTokenizer, pipeline, DistilBertForSequenceClassification

device = "cuda:0" if torch.cuda.is_available() else "cpu"
inputs = "Hello, my dog is cute"
if "CUDAExecutionProvider" in ort.get_available_providers() and device == "cuda:0":
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
else:
    providers = ["CPUExecutionProvider"]

def benchmark_inference(pipeline, inputs, n_runs=256):
    """简单性能测试函数"""
    with torch.no_grad():
        start = time.perf_counter()
        for _ in range(n_runs):
            _ = pipeline(inputs)
        end = time.perf_counter()
        avg = (end - start) / n_runs * 1000  # 毫秒
        return avg

def model_size(path):
    return os.path.getsize(path) / 1024 / 1024

# FP32 baseline
model_checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
classifier = bert_fp32(model_checkpoint, device)
baseline_cpu_ms = benchmark_inference(classifier, inputs)

# ONNX Runtime CPU
save_dir = "onnx/distilled-bert"
ort_classifier = bert_onnx_fp32(model_checkpoint, save_dir, providers, device)
ort_fp32_ms = benchmark_inference(ort_classifier, inputs)

# ONNX Runtime CPU - dynamic INT8
int8_dir = "onnx/int8-distilled-bert"
ort_int8_classifier = bert_quantize(model_checkpoint, int8_dir, providers, device)
ort_int8_ms = benchmark_inference(ort_int8_classifier, inputs)

# display
data = {
    '模型': ['Pipeline FP32', 'ONNXRuntime FP32', 'ONNXRuntime INT8'],
    '设备': ['CPU', 'CPU', 'CPU'],
    '延迟(ms)': [baseline_cpu_ms, ort_fp32_ms, ort_int8_ms],
    'onnx_size(MB)': [0, model_size('onnx/distilled-bert/model.onnx'), model_size('onnx/int8-distilled-bert/model_quantized.onnx')]
}

df = pd.DataFrame(data)

# Format the '延迟(ms)' column
df['延迟(ms)'] = df['延迟(ms)'].map('{:.2f}'.format)
df['onnx_size(MB)'] = df['onnx_size(MB)'].map('{:.2f}'.format)

display(df)

Device set to use cpu
The model distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
Device set to use cpu
The model distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
Device set to use cpu


,模型,设备,延迟(ms),onnx_size(MB)
0,Pipeline FP32,CPU,14.64,0.00
1,ONNXRuntime FP32,CPU,7.69,255.52
2,ONNXRuntime INT8,CPU,4.09,64.25
